# Using Libra built-in functions to compute overlaps in Kohn-Sham orbital basis

In this super tutorial, we will learn how to use the Libra's built-in functions to compute the overlap integrals using different methods.

## Table of contents
<a name="toc"></a>
1. [Importing needed libraries](#1)

2. [Overview of required files](#2)

3. [Computing KS overlap integrals using Gaussian `cube` files](#3)

4. [How to use Libint functions in Libra?](#4)

   4.1. [Creating an integration shell](#4.1)
  
   4.2. [Computing the overlap integral using `compute_overlaps` function](#4.2)
   
   4.3. [Computing moment integrals using `compute_emultipole3` function](#4.3)
   
   4.4. [Let's build a function](#4.4)
   
   4.5. [Computing Kohn-Sham molecular orbital overlaps using molden files](#4.5)
 

### A. Learning objectives

* To work with cube files and use them to compute MO time-overlaps
* To construct shells of AO functions
* To compute overlaps and multipoles in atomic orbitals basis using libint2 functions
* To read molden files and use them for AO integral calculations


### B. Use cases

* Working with cube files
* working with molden files
* Using libint2 functions to computing integrals
* Computing time-overlaps of AOs and MOs


### C. Functions

- `libra_py`
  - `cube_file_methods`
    - [`grid_volume`](#grid_volume-1)
    - [`integrate_cube`](#integrate_cube-1)
    - [`read_cube`](#read_cube-1)
  - `molden_methods`
    - [`eigenvectors_molden`](#eigenvectors_molden-1)
    - [`molden_file_to_libint_shell`](#molden_file_to_libint_shell-1)
  - `packages`
    - `cp2k`
      - `methods`
        - [`generate_translational_vectors`](#generate_translational_vectors-1)
        - [`resort_molog_eigenvectors`](#resort_molog_eigenvectors-1)
  - `workflows`
    - `nbra`
      - `step2`
        - [`component_to_index`](#component_to_index-1)

- `liblibra::liblibint2_wrappers`
  - [`add_to_shell`](#add_to_shell-1)
  - [`compute_emultipole3`](#compute_emultipole3-1)
  - [`compute_overlaps`](#compute_overlaps-1)
  - [`initialize_shell`](#initialize_shell-1)
  - [`nbasis`](#nbasis-1)
  - [`print_shell`](#print_shell-1)

## 1. Importing needed libraries <a name="1"></a>
[Back to TOC](#toc)

Let's import the following modules.

In [1]:
import os
import numpy as np
import scipy.sparse as sp
import time
from liblibra_core import *

from libra_py import units
from libra_py import data_conv
from libra_py import molden_methods
from libra_py import cube_file_methods
import libra_py.packages.cp2k.methods as CP2K_methods
from libra_py.workflows.nbra import step2

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

## 2. Overview of required files 
<a name="2"></a>[Back to TOC](#toc)

The files that we will be using for this super tutorial are as follows:

* `HOMO.cube`, `LUMO.cube`, `TiO2_unit_cell.molden`, `St_ks_1200.npz`, and a set of log files, `all_logfiles/`, that contain the TD-DFT data from CP2K calculation for the unit cell of TiO2 (see [this tutorial](../7_step2_cp2k/1_DFT/2_hpc/1_example_TiO2)). 

You need to first untar the file `data.tar.bz2` to extract these files.

In [2]:
#!tar -xf data.tar.bz2

## 3. Computing KS overlap integrals using Gaussian `cube` files
<a name="3"></a>[Back to TOC](#toc)

In this part we can see how we can use the `cube_file_methods` functions to read the data in cube files and integrate them. This approach was first used to interface CP2K, Gaussian 09, and DFTB+ with Libra. Currently, this approach is deprecated but it can be useful for small systems. These files contain the information for a cubic grid of the simulation cell. They can be large for large systems up to a couple of GBs.

Let's read a sample cube file for a TiO2 unit cell:
<a name="read_cube-1"></a>

In [3]:
help(cube_file_methods.read_cube)

Help on function read_cube in module libra_py.cube_file_methods:

read_cube(filename: str)
    Read scalar field data (e.g., wavefunction or electron density) from a Gaussian
    .cube file and return it as a flattened 1D NumPy array.
    
    This function parses the cube file header to determine where the volumetric
    data begins, then reads all grid values and stores them in a single array.
    The ordering of values follows the convention used in the cube file
    (typically x fastest, then y, then z).
    
    Parameters
    ----------
    filename : str
        Path to the .cube file.
    
    Returns
    -------
    isovalues : numpy.ndarray
        1D array containing the scalar field values on the grid.
    
    Notes
    -----
    - The number of atoms is read from the third line of the file. Its absolute
      value is used to handle Gaussian cube files where this number may be negative.
    - The function skips:
        * 2 comment lines
        * 1 line with number of at

In [4]:
homo_cube = cube_file_methods.read_cube('HOMO.cube')
print(homo_cube)

[-3.1842e-16 -6.2411e-17  3.9750e-16 ...  6.4707e-13  1.0841e-12
  1.5415e-12]


This is how we can compute the volume element $dv$ which is used in integration:
<a name="grid_volume-1"></a>

In [5]:
help(cube_file_methods.grid_volume)

Help on function grid_volume in module libra_py.cube_file_methods:

grid_volume(filename: str)
    Compute the volume element (voxel volume) of a grid cell from a Gaussian
    .cube file.
    
    The cube file defines the volumetric grid using three lattice vectors
    (one per axis), given in lines 4–6 of the file. Each vector corresponds
    to the spacing and direction of the grid along x, y, and z. The volume
    of a single grid cell is the absolute value of the determinant of these
    three vectors.
    
    Parameters
    ----------
    filename : str
        Path to the .cube file.
    
    Returns
    -------
    dv : float
        Volume of a single grid cell (voxel) in Bohr³.
    
    Notes
    -----
    - Lines 4–6 of the cube file contain:
        * number of grid points along each axis (first column)
        * corresponding lattice vector components (remaining columns)
    - Only the vector components are used here.
    - The total grid volume would be `dv * Nx * Ny * N

In [6]:
volume_element = cube_file_methods.grid_volume('HOMO.cube')
print(volume_element)

0.0026481024175524755


And we can integrate between cubes using:
<a name="integrate_cube-1"></a>

In [7]:
help(cube_file_methods.integrate_cube)

Help on function integrate_cube in module libra_py.cube_file_methods:

integrate_cube(cube_A, cube_B, grid_volume)
    Compute the numerical integral of the product of two scalar fields
    defined on the same volumetric grid (e.g., wavefunctions from .cube files).
    
    The integral is approximated as a discrete sum over all grid points:
        ∫ A(r) B(r) dV ≈ Σ_i A_i * B_i * dv
    where dv is the volume of a single grid cell (voxel).
    
    Parameters
    ----------
    cube_A, cube_B : numpy.ndarray
        1D arrays containing the volumetric data (e.g., wavefunctions or
        densities) sampled on the same grid. These are typically obtained
        from a cube file reader (e.g., `read_cube`).
        Both arrays must have the same shape and ordering.
    
    grid_volume : float
        Volume of a single grid cell (voxel), typically computed using
        `grid_volume`. Units are usually Bohr³.
    
    Returns
    -------
    integral : float
        Numerical approxima

In [8]:
int_homo_homo = cube_file_methods.integrate_cube(homo_cube, homo_cube, volume_element)
print(int_homo_homo)

1.0000023905586597


You can see the normalization of the HOMO cube file. 

## Excercise: 

> Try this for `LUMO.cube` and compute the overlap between HOMO and LUMO levels.

## 4. How to use Libint functions in Libra? 
<a name="4"></a>[Back to TOC](#toc)

There are lots of C++ implementation of the Libint code inside Libra. We have to first provide a (set) of integration shells.

### 4.1 Creating an integration shell
<a name="4.1"></a>[Back to TOC](#toc)

This shell includes the coordinate, spherical or Cartesian coordinate representation (via a boolean flag called `is_spherical`), angular momentum value, exponents and contraction coefficients of the Gaussian type orbital basis functions (see [this](https://www.cp2k.org/basis_sets)):

$$\varphi_i(\vec{r}) = R_i(r) \cdot Y_{l_i, m_i}(\theta, \phi)$$

$$R_i(r) = r^{l_i} \sum_{j=1}^{N} c_{ij} \cdot \exp(-\alpha_j \cdot r^2)$$

The functions used for this purpose are <a name="initialize_shell-1"></a> `initialize_shell` and <a name="add_to_shell-1"></a> `add_to_shell` which are part of `liblibra_core` module. The first one initializes an integration shell, and the other adds more shells to the initialized shell. This gives us the flexibility when reading turning the `molden` file into a set of integration shells. The coordinate are required to be defined as `VECTOR` type and the values of exponents and contraction coefficients are turned into C++ double type using `Py2Cpp_double` function. The exponents and the coefficients are passed as a `list` to both functions.

In [9]:
# Define the coordinate
coord = VECTOR(1.0, 0.000, 0.000)
# Spherical coordinate flag
is_spherical = True
# Angular momentum value
l_val = 0  # s-type orbital
# Exponent
exp = Py2Cpp_double([4.0])
# Coefficient
coeff = Py2Cpp_double([-0.5])
# Initialize a shell
shell = initialize_shell( int(l_val), is_spherical, exp, coeff, coord)

# Now add whatever remained from the basis set to this shell 
# as many times as needed
add_to_shell(shell, int(l_val), is_spherical, exp, coeff, coord)
add_to_shell(shell, int(l_val), is_spherical, exp, coeff, coord)
add_to_shell(shell, int(l_val), is_spherical, exp, coeff, coord)

You can check the information about the basis set via <a name="nbasis-1"></a> `nbasis(shell)` and <a name="print_shell-1"></a> `print_shell(shell)` function:

In [10]:
print('number of basis functions for this shell:',nbasis(shell))
# Let's print the shell itself
print_shell(shell)

number of basis functions for this shell: 4

	Shells are:
Shell:( O={1,0,0}
   {l=0,sph=1}
  4 -2.01584


 The shell size is:
4
Shell:( O={1,0,0}
   {l=0,sph=1}
  4 -2.01584


 The shell size is:
4
Shell:( O={1,0,0}
   {l=0,sph=1}
  4 -2.01584


 The shell size is:
4
Shell:( O={1,0,0}
   {l=0,sph=1}
  4 -2.01584


 The shell size is:
4


### 4.2 Computing the overlap integral using `compute_overlaps` function
<a name="4.2"></a>[Back to TOC](#toc)

Now let's compute the atomic orbital overlap matrix. This is done using the <a name="compute_overlaps-1"></a> `compute_overlaps` function which computes the overlap between two different integration shells and return a `MATRIX`. The number of processors is defined in `nprocs`.

In [11]:
nprocs = 4 # Set the number of processors
A = compute_overlaps(shell, shell, nprocs) # Compute the overlap matrix
A.show_matrix()

1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   



For practical purposes, turn the `MATRIX` into a `numpy` array using `data_conv.MATIX2nparray` function.

In [12]:
# Or turn it into a numpy array
data_conv.MATRIX2nparray(A).real

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

### 4.3 Computing moment integrals using `compute_emultipole3` function 
<a name="4.3"></a>[Back to TOC](#toc)

The components that are retrieved via the <a name="compute_emultipole3-1"></a> `compute_emultipole3` function are as follows:

```python
    all_components = ['S', 'x', 'y', 'z', 'x2', 'xy', 'xz', 'y2', 'yz', 'z2',
                      'x3', 'x2y', 'x2z', 'xy2', 'xyz', 'xz2', 'y3', 'y2z', 'yz2', 'z3']
```

In [13]:
A = compute_emultipole3(shell, shell, 4)
# We have 20 components from overlap S, to dipole and quadrupole
print(len(A))

20


There are 20 components of integrals. This also includes overlap integral, `S`. They are stored in the order shown in `all_components` above.

In [14]:
# Let's print overlap, x, y, z, and x2
print('overlap:')
A[0].show_matrix()

overlap:
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   



In [15]:
print('x:')
A[1].show_matrix()

x:
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   
1.0000000   1.0000000   1.0000000   1.0000000   



In [16]:
print('y:')
A[2].show_matrix()

y:
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   



In [17]:
print('z:')
A[3].show_matrix()

z:
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   



In [18]:
print('x2:')
A[4].show_matrix()

x2:
1.0625000   1.0625000   1.0625000   1.0625000   
1.0625000   1.0625000   1.0625000   1.0625000   
1.0625000   1.0625000   1.0625000   1.0625000   
1.0625000   1.0625000   1.0625000   1.0625000   



### 4.4 Let's build a function 
<a name="4.4"></a>[Back to TOC](#toc)

We can try to make a function like as follows (this is not part of Libra but just a test function):

In [19]:
def create_shell_atom(coord, is_spherical, l_vals, exps, coeffs):
    """
    This function creates a set of main shells, each are separate, for all atoms in the system.
    """
    a = VECTOR(coord[0], coord[1], coord[2])
    for c1 in range(len(l_vals)):
        if c1==0:
            shell = initialize_shell(int(l_vals[c1]), is_spherical, Py2Cpp_double(exps[c1]), Py2Cpp_double(coeffs[c1]), a)
        else:
            add_to_shell(shell, int(l_vals[c1]), is_spherical, Py2Cpp_double(exps[c1]), Py2Cpp_double(coeffs[c1]), a)
            
    return shell

With this, we can create a set of integration shell related to an atom. This can be useful when you want to interface another software package, like **ORCA** or **OpenMolcas** or **QChem**, with Libra.

In [20]:
# Coordinate
coord = [1.0, 0.0, 0.0]
# Define the angular momentum values as appear in a basis set file
l_vals = [0,0,1,1,2]
# The list of exponents for each angular momentum value
exps = [[12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761],[12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761],
       [12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761],[12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761],
       [12.015955,5.108150,2.048398,0.832382,0.352316,0.142977,0.046761]]
# And their corresponding coefficients
coeffs = [[-0.060191,-0.129598,0.118176,0.462964,0.450354,0.092716,-0.000256],[0.065739,0.110886,-0.053732,-0.572671,0.186760,0.387201,0.003826],
         [0.036544,0.120928,0.251094,0.352640,0.294709,0.173040,0.009726],[-0.034211,-0.120620,-0.213719,-0.473675,0.484848,0.717466,0.032499],
         [0.014807,0.068186,0.290576,1.063344,0.307656,0.318347,-0.005772]]
# Create the integration shell
shell = create_shell_atom(coord, is_spherical, l_vals, exps, coeffs)
print('number of basis sets for this shell:', nbasis(shell))

number of basis sets for this shell: 13


Let's print the shell information

In [21]:
print_shell(shell)


	Shells are:
Shell:( O={1.0000000,0.0000000,0.0000000}
   {l=0,sph=1}
  12.015955 -0.28449995
  5.1081500 -0.32249868
  2.0483980 0.14819104
  0.83238200 0.29547527
  0.35231600 0.15082913
  0.14297700 0.015788331
  0.046761000 -1.8853185e-05


 The shell size is:
5
Shell:( O={1.0000000,0.0000000,0.0000000}
   {l=0,sph=1}
  12.015955 0.72562543
  5.1081500 0.64438457
  2.0483980 -0.15734917
  0.83238200 -0.85352822
  0.35231600 0.14606758
  0.14297700 0.15397733
  0.046761000 0.00065800393


 The shell size is:
5
Shell:( O={1.0000000,0.0000000,0.0000000}
   {l=1,sph=1}
  12.015955 1.1921517
  5.1081500 1.3541725
  2.0483980 0.89726731
  0.83238200 0.40883918
  0.35231600 0.11664777
  0.14297700 0.022184355
  0.046761000 0.00030839447


 The shell size is:
5
Shell:( O={1.0000000,0.0000000,0.0000000}
   {l=1,sph=1}
  12.015955 -1.0623948
  5.1081500 -1.2857933
  2.0483980 -0.72699821
  0.83238200 -0.52276451
  0.35231600 0.18268099
  0.14297700 0.087560117
  0.046761000 0.00098095033




In [22]:
# Now let's compute the atomic orbital overlap matrix
nprocs = 4 # Set the number of processors
A = compute_overlaps(shell, shell, nprocs) # Compute the overlap matrix
A.show_matrix()

1.0000000   -0.088210134  0.0000000   -8.2888693e-18  0.0000000   0.0000000   1.2754768e-17  0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   
-0.088210134  1.0000000   0.0000000   -1.6600822e-17  0.0000000   0.0000000   3.2941720e-18  0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   
0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   0.0000000   -1.6738102e-18  0.0000000   0.0000000   0.0000000   
-8.2888693e-18  -1.6600822e-17  0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   9.6637478e-19  0.0000000   0.0000000   -1.6738102e-18  0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   0.0000000   0.0000000   -1.6738102e-18  
0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.0000000   -1.5077029e-18  0.0000000   0.0000000   0.0000000   
1.2754768e-17

And now let's see the other integral components:

In [23]:
A = compute_emultipole3(shell, shell, 4)
# We have 20 components from overlap S, to dipole and quadrupole
print(len(A))
# Let's print overlap, x, y, z, and x2
print('x:')
A[1].show_matrix()

20
x:
1.0000000   -0.088210134  0.0000000   0.67340060  0.0000000   0.0000000   0.31935015  0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   0.0000000   
-0.088210134  1.0000000   0.0000000   0.39866373  0.0000000   0.0000000   1.2570647   0.0000000   5.5511151e-17  0.0000000   0.0000000   5.5511151e-17  0.0000000   
0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   0.0000000   0.60376629  0.0000000   0.0000000   0.0000000   
0.67340060  0.39866373  0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   -0.34858463  0.0000000   0.0000000   0.60376629  0.0000000   
0.0000000   0.0000000   0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   0.0000000   0.0000000   0.60376629  
0.0000000   0.0000000   0.17653908  0.0000000   0.0000000   1.0000000   0.0000000   0.0000000   0.0000000   0.52270802  0.0000000   0.0000000   0.0000000   
0.31935015  1.2570647   0.0000000   0.176

### 4.5 Computing Kohn-Sham molecular orbital overlaps using molden files 
<a name="4.5"></a>[Back to TOC](#toc)

Here, we perform the overlap calculations using the molecular orbital coefficients. In CP2K, the basis set is of Gaussian type orbitals which are not orthonormal. We need to compute the atomic orbital overlap matrix between different types of GTOs and then use the molecular orbital coefficients to compute the molecular orbital overlaps:

$$\langle\phi_i|\phi_j\rangle = \sum_{a_{i}=0}^{N}\sum_{b_{j}=0}^{N}c_{a_i}^*c_{b_j}^*\langle\psi_{a_{i}}|\psi_{b_{j}}\rangle$$

### How does a `molden` file look like  <a name="molden_file"></a>

This is how a `molden` file format looks like:
```
 [Molden Format]
 [Atoms] AU
 Ti       1      22       4.396705       4.396705       2.805490
 Ti       2      22       0.000000       0.000000       0.000000
 O        3       8       1.718408       7.075002       2.805490
 O        4       8       7.075002       1.718408       2.805490
 O        5       8       2.678297       2.678297       0.000000
 O        6       8       6.115113       6.115113       0.000000
 [GTO]
        1       0
                         s       6    1.00
                                                          7.88456993        0.00475058
                                                          3.89469846        0.49950386
                                                          1.51358883       -0.66499588
                                                          0.59676808       -0.72604457
                                                          0.22222213       -0.02901108
                                                          0.07707846        0.07517175
                         s       6    1.00
                                                          7.88456993       -0.00269070
....
 [MO]
Ene=   -2.1223307907E+00
Spin= Alpha
Occup=   2.0000000
     1 -7.01185407E-01
     2  3.22437005E-02
     3  9.91737658E-03
....
```

We have a sample molden file for a **periodic** structure, the TiO2 unit cell from the previous steps. The reason we select this is to show how to use the traslational vectors in computation of the molecular overlap integrations.
The main function in Libra that turns a `molden` file into a Libint shell using the above functions is `molden_file_to_libint_shell` in the `libra_py.molden_methods` module. It's documentation is as follows:

```python
def molden_file_to_libint_shell(molden_filename: str, is_spherical: bool, is_periodic=False,
                                cell=np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]]), R_vec=np.array([0, 0, 0])):
    """
    This function gets the molden file and returns the shell for use with
    libint to compute the atomic orbital overlaps.

    Args:

        molden_filename (string): The name of the molden file.

        is_spherical (bool): The Gaussian cartesian or spherical orbital flag.
        
        is_periodic (bool): Whether the cell is periodic or not.
        
        cell (nparray): If the cell is periodic, what is the cell dimensions.
        
        R_vec (nparray): What are the translational vector

    Returns:

        shell (shell data type from libint): The shell containing the exponents and contraction
                                             coefficients for each atom, their coordinates in Bohr,
                                             and their angular momentum values.
        l_vals_all (list): The angular momentum values for all the atoms and basis set in order.
                           This is used later to resort the eigenvectors in molden file.

    """
```

Let's see how it operates. Before that, since this system is periodic, we need to define a set of translational vectors from CP2K `&CELL` input or from the cell information in the coordinate file (`POSCAR` or `CIF` etc):
```python
params['A_cell_vector'] = [4.6532721519, 0.0000000000, 0.0000000000]
params['B_cell_vector'] = [0.0000000000, 4.6532721519, 0.0000000000]
params['C_cell_vector'] = [0.0000000000, 0.0000000000, 2.9692029953]
params['periodicity_type'] = 'XYZ'
```
and the translational vectors are defined using <a name="generate_translational_vectors-1"></a> `CP2K_methods.generate_translational_vectors` function:
```python
# Set the origin for generating the translational vectors (for creating Bloch type functions)
origin = [0,0,0]
tr_vecs = params['translational_vectors'] = CP2K_methods.generate_translational_vectors(origin, [2,2,2],
                                                                                        params['periodicity_type'])
```
the variable `[2,2,2]` defines how many periodic images to reproduce in each of the X, -X, Y, -Y, Z, and -Z directions respectively. It can be changed and you can see that we are producing $(2\times2+1)^3-1=125-1=124$ translational vectors (excluding $(0,0,0)$ which is the central cell itself.

Let's try this for `[1,1,1]` i.e. $(2\times1+1)^3-1=26$ translational vectors.

In [24]:
params_1 = {}
params_1['A_cell_vector'] = [4.6532721519, 0.0000000000, 0.0000000000]
params_1['B_cell_vector'] = [0.0000000000, 4.6532721519, 0.0000000000]
params_1['C_cell_vector'] = [0.0000000000, 0.0000000000, 2.9692029953]
params_1['periodicity_type'] = 'XYZ'
origin = [0,0,0]
tr_vecs = params_1['translational_vectors'] = CP2K_methods.generate_translational_vectors(origin, [1,1,1],
                                                                                        params_1['periodicity_type'])
print('The translational vectors for the current periodic system are:\n')
print(tr_vecs)
print(F'Will compute the S^AO between R(0,0,0) and {tr_vecs.shape[0]} translational vectors')

The translational vectors for the current periodic system are:

[[-1 -1 -1]
 [-1 -1  0]
 [-1 -1  1]
 [-1  0 -1]
 [-1  0  0]
 [-1  0  1]
 [-1  1 -1]
 [-1  1  0]
 [-1  1  1]
 [ 0 -1 -1]
 [ 0 -1  0]
 [ 0 -1  1]
 [ 0  0 -1]
 [ 0  0  1]
 [ 0  1 -1]
 [ 0  1  0]
 [ 0  1  1]
 [ 1 -1 -1]
 [ 1 -1  0]
 [ 1 -1  1]
 [ 1  0 -1]
 [ 1  0  0]
 [ 1  0  1]
 [ 1  1 -1]
 [ 1  1  0]
 [ 1  1  1]]
Will compute the S^AO between R(0,0,0) and 26 translational vectors


Before going into calculations, let's increase the number of translational vectors to have more accuracy:

In [25]:
help(CP2K_methods.generate_translational_vectors)

Help on function generate_translational_vectors in module libra_py.packages.cp2k.methods:

generate_translational_vectors(origin, N, periodicity_type)
    This function generates the translational vectors for periodic systems.
    For monolayers the generated vectors does not add the orthogonal axis but
    for bulk all directions are added.
    
    Args:
        origin (list): The translational vectors are obtained with respect to this origin.
        N (list): An array that contains the number of cells to be considered in
                     each of the X, Y, and Z directions.
        periodicity_type (string): The periodicity type. It can only get these values:
                                   'XY', 'XZ', and 'YZ' for monolayers and 'XYZ' for bulk systems.
    
    Returns:
        translational_vectors (numpy array): The translational vectors for that system.



In [26]:
tr_vecs = CP2K_methods.generate_translational_vectors(origin, [2,2,2], params_1['periodicity_type'])

**Important note:** We need to turn the cell vectors to atomic units first using `units.Angst`:

In [27]:
cell = []
cell.append(params_1['A_cell_vector'])
cell.append(params_1['B_cell_vector'])
cell.append(params_1['C_cell_vector'])
cell = np.array(cell) * units.Angst

Then, we generate the integral shell and the angular momentum values related to each basis functions as they appear in the molden file. Let's consider the central cell for now i.e. `R_vec=np.array([0, 0, 0])`.
<a name="molden_file_to_libint_shell-1"></a>

In [28]:
help(molden_methods.molden_file_to_libint_shell)

Help on function molden_file_to_libint_shell in module libra_py.molden_methods:

molden_file_to_libint_shell(molden_filename: str, is_spherical: bool, is_periodic=False, cell=array([[0, 0, 0],
       [0, 0, 0],
       [0, 0, 0]]), R_vec=array([0, 0, 0]))
    This function gets the molden file and returns the shell for use with
    libint to compute the atomic orbital overlaps.
    
    Args:
    
        molden_filename (string): The name of the molden file.
    
        is_spherical (bool): The Gaussian cartesian or spherical orbital flag.
    
    Returns:
    
        shell (shell data type from libint): The shell containing the exponents and contraction
                                             coefficients for each atom, their coordinates in Bohr,
                                             and their angular momentum values.
        l_vals_all (list): The angular momentum values for all the atoms and basis set in order.
                           This is used later to resort t

In [29]:
shell_1, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array([0, 0, 0]))
print(shell_1)
print('number of basis functions:',nbasis(shell_1))
print(l_vals)

number of basis functions: 104
[0, 0, 0, 1, 1, 2, 2, 3, 0, 0, 0, 1, 1, 2, 2, 3, 0, 0, 1, 1, 2, 0, 0, 1, 1, 2, 0, 0, 1, 1, 2, 0, 0, 1, 1, 2]


 You can `print_shell(shell_1)` yourself (it would be a lengthy output).

Now, let's compute the overlap between shells using `compute_overlaps` function.

In [30]:
# number of processors to use
nprocs = 4
# Atomic orbital overlap matrix
AO = compute_overlaps(shell_1, shell_1, nprocs)
print(AO)

We need to turn this `MATRIX` object to `nparray` first using:

In [31]:
AO_numpy = data_conv.MATRIX2nparray(AO)
print(AO_numpy[0:10,0])

[ 1.00000000e+00+0.j  6.42868266e-02+0.j -2.08855695e-01+0.j
  5.20933915e-16+0.j -3.77792680e-17+0.j -3.77792680e-17+0.j
 -2.53972475e-16+0.j -1.08710819e-16+0.j -1.08710819e-16+0.j
  0.00000000e+00+0.j]


To generate the atomic orbital overlap matrix for all translational vectors we need a `for` loop as follows:

In [32]:
AO = compute_overlaps(shell_1, shell_1, nprocs)
for tr_vec in tr_vecs:
    shell_periodic, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array(tr_vec))
    AO += compute_overlaps(shell_1, shell_periodic, nprocs)
# Now turn it into numpy array
AO_numpy = data_conv.MATRIX2nparray(AO)

#### Sorting the indices

Different software use different convention to present the angular momentum components. The version of Libint that is installed via `conda` is for `Psi4` package and not CP2K. So we need to resort the eigenvectors (or the atomic orbital matrix itself) accordingly. This is done using <a name="resort_molog_eigenvectors-1"></a> `CP2K_methods.resort_molog_eigenvectors` function. 

In [33]:
help(CP2K_methods.resort_molog_eigenvectors)

Help on function resort_molog_eigenvectors in module libra_py.packages.cp2k.methods:

resort_molog_eigenvectors(l_vals)
    This function returns the resotring indices for resoting the MOLog
    eigenvectors according to this order:
    
    MOLog order (example for Cd atom):
    2s, 3s | 3py, 3pz, 3px | 4py, 4pz, 4px | 4d-2, 4d-1, 4d0, 4d+1, 4d+2 | 5d-2, 5d-1, 5d0, 5d+1, 5d+2 | ...
    However, the atomic orbital overla computed from the libint version of Psi4 is not ordered
    as above. The ordering is like this:
    2s, 3s | 3pz, 3px, 3py | 4pz, 4px, 4py | 4d0, 4d+1, 4d-1, 4d+2, 4d-2 | 5d0, 5d+1, 5d-1, 5d+2, 5d-2 | ...
    Therefore, we need to resort the eigenvectors to be able to use the code properly.
    
    Args:
    
        l_vals (list): A list containing the angular momentum values for atoms
                       in the order of the MOLog files.
    
    Returns:
    
        new_indices (numpy array): The new indices that needs to be used for reordering.



In [34]:
new_indices = CP2K_methods.resort_molog_eigenvectors(l_vals)
print(new_indices)

[np.int64(0), np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(3), np.int64(7), np.int64(8), np.int64(6), np.int64(11), np.int64(12), np.int64(10), np.int64(13), np.int64(9), np.int64(16), np.int64(17), np.int64(15), np.int64(18), np.int64(14), np.int64(22), np.int64(23), np.int64(21), np.int64(24), np.int64(20), np.int64(25), np.int64(19), np.int64(26), np.int64(27), np.int64(28), np.int64(30), np.int64(31), np.int64(29), np.int64(33), np.int64(34), np.int64(32), np.int64(37), np.int64(38), np.int64(36), np.int64(39), np.int64(35), np.int64(42), np.int64(43), np.int64(41), np.int64(44), np.int64(40), np.int64(48), np.int64(49), np.int64(47), np.int64(50), np.int64(46), np.int64(51), np.int64(45), np.int64(52), np.int64(53), np.int64(55), np.int64(56), np.int64(54), np.int64(58), np.int64(59), np.int64(57), np.int64(62), np.int64(63), np.int64(61), np.int64(64), np.int64(60), np.int64(65), np.int64(66), np.int64(68), np.int64(69), np.int64(67), np.int64(71), np.int64(72), n

Now, we read the eigenvectors. Due to a specific writing format of the eigenvectors in molden files, we need to know what is the number of basis functions that appear in an integration shells i.e. all number of atomic orbitals, which is done using `nbasis` function from `liblibra_core` module.
<a name="eigenvectors_molden-1"></a>

In [35]:
help(molden_methods.eigenvectors_molden)

Help on function eigenvectors_molden in module libra_py.molden_methods:

eigenvectors_molden(molden_filename: str, nbasis: int, l_vals: list)
    This functions read the molecular orbitals eigenvectors and resort
    them so that we can use it with the libint.
    
    Args:
    
        molden_filename (string): The name of molden file.
    
        nbasis (integer): This is obtained from the liblibra_core.nbasis. It is necessary
                          since sometime the molden files will not append all the values
                          and there would be some values missing. It is better to use a
                          higher value of NDIGITS in the CP2K input file to have most of
                          the values.
    
        l_vals (list): All the angular momentum values return by the molden_file_to_libint_shell
                       function for the molden file.
    
    Returns:
    
        eigenvectors (list): All the sorted eigenvectors for use with the libint co

In [36]:
number_of_basis_functions = nbasis(shell_1)
print(F"Number of basis functions = {number_of_basis_functions}")
eigenvectors, energies = molden_methods.eigenvectors_molden('TiO2_unit_cell.molden', number_of_basis_functions, l_vals)
print(F"Number of MOs = {len(eigenvectors)}")
print("Eigenvectors shape = ", eigenvectors[0].shape)

Number of basis functions = 104
Number of MOs = 74
Eigenvectors shape =  (104,)


For unrestricted spin calculations, the eigenvectors are sorted in a different way. Please be careful of parsing the `molden` file. It might be the case that the alpha and beta orbitals are written either in a sequential format or they are mixed. The way CP2K outputs these eigenvectors is that the first half of the eigenvectors are the alpha MOs and the next half are beta MOs.

Now, let's sort the eigenvectors according to the sorting key computed above:

In [37]:
eigenvectors_1 = []
for j in range(len(eigenvectors)):
    # the new and sorted eigenvector
    eigenvector_1 = eigenvectors[j]
    eigenvector_1 = eigenvector_1[new_indices]
    # append it to the eigenvectors list
    eigenvectors_1.append(eigenvector_1)
eigenvectors_1 = np.array(eigenvectors_1)

And finally, computing the MO overlap matrix:

In [38]:
MO_overlap = np.linalg.multi_dot([eigenvectors_1, AO_numpy, eigenvectors_1.T])
print(np.diag(MO_overlap))

[0.99999961+0.j 1.00000041+0.j 0.99999999+0.j 1.        +0.j
 1.00000031+0.j 1.00000029+0.j 0.99998989+0.j 1.00001294+0.j
 0.99995745+0.j 1.00003095+0.j 1.00000712+0.j 1.00000716+0.j
 0.99999744+0.j 0.99999625+0.j 0.99999708+0.j 1.00001662+0.j
 1.00001658+0.j 0.9999872 +0.j 0.99998869+0.j 0.99908439+0.j
 1.00007521+0.j 0.9999783 +0.j 0.99997998+0.j 1.00001046+0.j
 0.9999867 +0.j 1.00001724+0.j 0.99999472+0.j 1.00006086+0.j
 1.00006085+0.j 1.000052  +0.j 0.99999583+0.j 0.99958757+0.j
 0.99958764+0.j 0.9995215 +0.j 1.00021321+0.j 1.00004889+0.j
 1.00005065+0.j 0.99959955+0.j 1.00496174+0.j 0.88632026+0.j
 1.01034228+0.j 1.00116205+0.j 1.0011538 +0.j 1.09927545+0.j
 0.98099997+0.j 1.01329743+0.j 1.05242962+0.j 1.05242983+0.j
 1.00020224+0.j 1.00008737+0.j 0.97469219+0.j 1.04710326+0.j
 0.99587122+0.j 0.95245193+0.j 1.02667602+0.j 1.02667578+0.j
 1.00006537+0.j 1.00498198+0.j 0.99969375+0.j 0.9996906 +0.j
 1.01092995+0.j 0.99595327+0.j 0.99595416+0.j 0.98170708+0.j
 1.00778589+0.j 1.007790

Let's compute the determinant of this matrix to check its orthonormality:

In [39]:
print(np.linalg.det(MO_overlap))

(0.9068820174820696+0j)


It's still not orthonormal but close to it. To further increase the accuracy, you need to include more number of translational vectors. Let's continue with `[3,3,3]` for translational vectors:

In [40]:
tr_vecs = CP2K_methods.generate_translational_vectors(origin, [3,3,3], params_1['periodicity_type'])
AO = compute_overlaps(shell_1, shell_1, nprocs)
for tr_vec in tr_vecs:
    shell_periodic, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array(tr_vec))
    AO += compute_overlaps(shell_1, shell_periodic, nprocs)
# Now turn it into numpy array
AO_numpy = data_conv.MATRIX2nparray(AO)
MO_overlap = np.linalg.multi_dot([eigenvectors_1, AO_numpy, eigenvectors_1.T])

In [41]:
print(np.linalg.det(MO_overlap))

(1.000740369408316+0j)


In [42]:
print(np.diag(MO_overlap))

[1.        +0.j 1.00000001+0.j 1.        +0.j 1.00000001+0.j
 1.00000001+0.j 1.00000001+0.j 1.00000002+0.j 1.00000003+0.j
 1.        +0.j 1.00000023+0.j 0.99999996+0.j 0.99999998+0.j
 1.0000001 +0.j 1.00000006+0.j 1.00000006+0.j 1.0000001 +0.j
 1.00000005+0.j 0.99999993+0.j 0.99999999+0.j 0.99999434+0.j
 1.00000036+0.j 0.99999986+0.j 0.99999986+0.j 1.00000003+0.j
 0.99999988+0.j 0.99999993+0.j 1.        +0.j 1.00000006+0.j
 1.00000005+0.j 1.00000055+0.j 0.99999992+0.j 0.99999897+0.j
 0.99999906+0.j 0.99999887+0.j 0.99999947+0.j 1.00000014+0.j
 1.00000027+0.j 0.99999987+0.j 1.00004873+0.j 0.99966431+0.j
 1.00001186+0.j 1.00000076+0.j 1.00000059+0.j 1.0002019 +0.j
 0.99994581+0.j 1.00000945+0.j 1.00006596+0.j 1.000066  +0.j
 0.99999981+0.j 0.9999997 +0.j 0.99992055+0.j 1.00047491+0.j
 0.99999309+0.j 1.00009738+0.j 1.00002634+0.j 1.00002643+0.j
 0.99999983+0.j 1.00003122+0.j 0.99999924+0.j 0.99999906+0.j
 1.0000136 +0.j 0.99998816+0.j 0.99998824+0.j 0.99994193+0.j
 1.0000264 +0.j 1.000026

You can see that the determinant is `1.00074` (accuracy up to $10^{-3}$). Let's try one more by increasing `[4,4,4]`:

In [43]:
tr_vecs = CP2K_methods.generate_translational_vectors(origin, [4,4,4], params_1['periodicity_type'])
AO = compute_overlaps(shell_1, shell_1, nprocs)
for tr_vec in tr_vecs:
    shell_periodic, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array(tr_vec))
    AO += compute_overlaps(shell_1, shell_periodic, nprocs)
# Now turn it into numpy array
AO_numpy = data_conv.MATRIX2nparray(AO)
MO_overlap = np.linalg.multi_dot([eigenvectors_1, AO_numpy, eigenvectors_1.T])
print(np.linalg.det(MO_overlap))

(0.9999897428194451+0j)


which has a higher accuracy of $10^{-4}$.

You can do the same thing to compute the overlap between different geoemtries using the same procedure. This procedure is implemented in step 2 of Libra for computation of the time-overlap of molecular orbitals of two consecutive geometries. See [this tutorial](../7_step2_cp2k).

#### Computing the integrals using `compute_emultipole3` for periodic system

Let's try the `compute_emultipole3` function and compute the dipole moment operator matrix in the molecular orbital basis: 

In [44]:
tr_vecs = CP2K_methods.generate_translational_vectors(origin, [4,4,4], params_1['periodicity_type'])
A = compute_emultipole3(shell_1, shell_1, nprocs)
for tr_vec in tr_vecs:
    shell_periodic, l_vals = molden_methods.molden_file_to_libint_shell('TiO2_unit_cell.molden', is_spherical=True, is_periodic=True,
                                cell=cell, R_vec=np.array(tr_vec))
    A_periodic = compute_emultipole3(shell_1, shell_periodic, nprocs)
    # Since there are 20 components
    for i in range(20):
        A[i] += A_periodic[i]

You can simply use a list like `['S','x','y','z','x2']` and turn it into a set of indices and labels using `step2.component_to_index` function as follows:
<a name="component_to_index-1"></a>

In [45]:
emultipole_index, emultipole_labels = step2.component_to_index(['S','x','y','z','x2'])
print(emultipole_index, emultipole_labels)

[0, 1, 2, 3, 4] ['S', 'x', 'y', 'z', 'x2']


In [46]:
# Now turn each component into numpy array
MO_matrices = []
for i in emultipole_index:
    A_numpy = data_conv.MATRIX2nparray(A[i])
    MO_matrix = np.linalg.multi_dot([eigenvectors_1, A_numpy, eigenvectors_1.T])
    print('The first 10 diagonal element of the', emultipole_labels[i], 'component:')
    print(np.diag(MO_matrix)[0:10])
    MO_matrices.append(MO_matrix)

The first 10 diagonal element of the S component:
[1.0000055 +0.j 1.00000208+0.j 1.00008344+0.j 0.999991  +0.j
 0.99994302+0.j 1.00002374+0.j 1.00000009+0.j 0.9999769 +0.j
 1.00228201+0.j 0.99594676+0.j]
The first 10 diagonal element of the x component:
[2.2172853 +0.j 2.22771187+0.j 4.14485462+0.j 0.38166782+0.j
 0.3470383 +0.j 4.13320098+0.j 2.24931283+0.j 2.22450337+0.j
 4.2964396 +0.j 4.25311259+0.j]
The first 10 diagonal element of the y component:
[2.21705454+0.j 2.22733692+0.j 4.14488107+0.j 0.38152321+0.j
 0.3498728 +0.j 4.13322   +0.j 2.24901569+0.j 2.22440364+0.j
 4.2339837 +0.j 4.23265407+0.j]
The first 10 diagonal element of the z component:
[1.39981374+0.j 1.39757712+0.j 2.63133472+0.j 0.16794486+0.j
 0.21699069+0.j 2.57122458+0.j 1.40422962+0.j 1.40112222+0.j
 1.33548772+0.j 1.43485242+0.j]
The first 10 diagonal element of the x2 component:
[10.2921404 +0.j 10.37182474+0.j 18.89035284+0.j  2.5328229 +0.j
  2.33372665+0.j 18.81997022+0.j 10.31201061+0.j 10.15659408+0.j
 24

Let's check the determinant of the MO overlap matrix using emultipole3 approach:

In [47]:
print('Determinant of the overlap matrix from compute_emultipole3 function:', np.linalg.det(MO_matrices[0]))

Determinant of the overlap matrix from compute_emultipole3 function: (0.040283320697258915+0j)
